# Results for model: mistralai/mistral-small-3.1-24b-instruct-2503

In [ ]:
import polars as pl
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Load the data
data = pl.read_parquet('./jane_street_data/train.parquet')

# Function to calculate the global top quantile
def calculate_top_quantile(df, feature, target, quantile=0.15):
    top_quantile = df.select(target).quantile(quantile)
    df = df.with_column(
        pl.when(pl.col(feature) > top_quantile[0, 0])
        .then(1)
        .otherwise(0)
        .alias('top_feature_00')
    )
    return df

# Calculate the top quantile for 'feature_00' relative to 'responder_6'
global_top_quantile = data.groupby('date_id').agg([
    calculate_top_quantile
]).filter(pl.col('date_id') >= 0)

# Convert to Pandas for XGBoost compatibility
import pandas as pd

data_pd = data.to_pandas()

# Split the data
X = data_pd[['feature_00', 'feature_01', 'feature_02']]
y = data_pd['responder_6']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the XGBoost Regressor
model = XGBRegressor(objective='reg:squarederror', use_label_encoder=False, eval_metric='rmse')
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f'Root Mean Squared Error: {rmse}')